# RAG Summarization Evaluation

Evaluates three RAG configurations on the student feedback dataset:
- **Baseline RAG**: global top-K cosine similarity, no filtering
- **Supervised RAG**: category-filtered top-K
- **Supervised + Unsupervised RAG (Ours)**: category-filtered, cluster-stratified

**Metrics**
- Faithfulness = N_supported / N_total (NLI-based per sentence)
- Cluster Coverage = C_represented / C_available (deterministic)

**Note**: Set `USE_OPENAI = True` and provide `OPENAI_API_KEY` to use GPT-4o / GPT-4o-mini as described in the manuscript. By default the notebook uses free local models (DistilBART + DeBERTa NLI).

In [1]:
# ── Cell 1: Configuration ────────────────────────────────────────────────────

USE_OPENAI    = False   # True → GPT-4o generation + GPT-4o-mini judging (~$2-4)
OPENAI_API_KEY = ""     # Set here OR export OPENAI_API_KEY in your shell

K = 5  # retrieved documents per query (5 ensures coverage stays below 1.0)

# Paths (relative to this notebook's location)
import os
NOTEBOOK_DIR   = os.path.abspath('')
CLUSTERING_DIR = os.path.join(NOTEBOOK_DIR, '..', 'clustering-module')
DATA_DIR       = os.path.join(CLUSTERING_DIR, 'data')
EMB_DIR        = os.path.join(DATA_DIR, 'embeddings')
DATASET_CSV    = os.path.join(DATA_DIR, 'dataset_tsne_clustering.csv')

print(f'Dataset  : {DATASET_CSV}')
print(f'Exists   : {os.path.exists(DATASET_CSV)}')
N_QUERIES_PER_CAT = 3   # queries sampled per category (natural variance)


Dataset  : /Users/yerasap/Downloads/RA/t-sne generator/../clustering-module/data/dataset_tsne_clustering.csv
Exists   : True


In [2]:
# ── Cell 2: Install dependencies (run once) ──────────────────────────────────
# Uncomment if any package is missing:
# !pip install transformers>=4.35 sentence-transformers nltk tqdm sentencepiece accelerate
# !pip install openai  # only needed when USE_OPENAI = True

In [3]:
# ── Cell 3: Imports ──────────────────────────────────────────────────────────
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import nltk
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

if USE_OPENAI:
    import os as _os
    _key = OPENAI_API_KEY or _os.getenv('OPENAI_API_KEY', '')
    if not _key:
        raise ValueError('Set OPENAI_API_KEY or provide it in Cell 1.')
    from openai import OpenAI
    openai_client = OpenAI(api_key=_key)
    print('OpenAI client initialised.')
else:
    from transformers import pipeline
    from sentence_transformers import CrossEncoder
    print('Using local models (DistilBART + DeBERTa-NLI).')

Using local models (DistilBART + DeBERTa-NLI).


In [4]:
# ── Cell 4: Load dataset + align cluster labels ──────────────────────────────
#
# dataset_tsne_clustering.csv has the pre-computed HDBSCAN cluster_label column
# but its row order differs from the NPZ embedding files (which are ordered as
# first-671-train-originals + 288-test).  We align by feedback text matching.

import sys
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.join(CLUSTERING_DIR, '..', 'clustering-module'))

# 1. Reconstruct the 959-sample ordered dataframe (matches NPZ order exactly)
_df_orig = pd.read_csv(os.path.join(DATA_DIR, 'dataset.csv'))
_le      = LabelEncoder()
_labels  = _le.fit_transform(_df_orig['category'].values)
_df_tr, _df_te, _, _ = train_test_split(
    _df_orig, _labels, test_size=0.3, random_state=42, stratify=_labels
)
df = pd.concat([_df_tr.iloc[:671], _df_te]).reset_index(drop=True)

# 2. Get cluster labels from dataset_tsne_clustering.csv via text matching
_df_clust   = pd.read_csv(DATASET_CSV)
_clust_map  = dict(zip(_df_clust['feedback'].str.strip(),
                        _df_clust['cluster_label']))
df['cluster_label'] = df['feedback'].str.strip().map(_clust_map).fillna(-1).astype(int)

# 3. Text column for DistilBART / NLI
df['_text'] = df['Cleaned Feedback'].fillna(df['feedback']).astype(str)

print(f'Loaded {len(df)} samples, {df["category"].nunique()} categories')
print(f'Cluster labels: {sorted(df["cluster_label"].unique())}')
assert len(df) == 959, 'Expected 959 samples'


Loaded 959 samples, 10 categories
Cluster labels: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [5]:
# ── Cell 5: Build corpus embeddings for retrieval ────────────────────────────
#
# PRIMARY: use pre-computed Concat+PCA embeddings (53 dims, 66.7% variance).
#          These match the NPZ row order (671 train originals + 288 test).
# FALLBACK: encode with sentence-transformers if NPZ files are unavailable.

from src.fusion import load_individual_embeddings, build_concat_pca_fusion

_TEST_NPZ  = os.path.join(EMB_DIR, 'tsne_stage3_Test_individual.npz')
_TRAIN_NPZ = os.path.join(EMB_DIR, 'tsne_stage3_Train_individual.npz')

if os.path.exists(_TEST_NPZ) and os.path.exists(_TRAIN_NPZ):
    print('Loading pre-computed NPZ embeddings...')
    _emb = load_individual_embeddings(test_npz=_TEST_NPZ, train_npz=_TRAIN_NPZ)
    corpus_embeddings, _ = build_concat_pca_fusion(_emb)
    print(f'Concat+PCA corpus embeddings: {corpus_embeddings.shape}')
    assert len(corpus_embeddings) == len(df), \
        f'Alignment error: {len(corpus_embeddings)} embeddings vs {len(df)} rows'
    print('Alignment OK.')
else:
    print('NPZ files not found — falling back to sentence-transformer encoding.')
    print('Loading sentence encoder (sentence-transformers/multi-qa-mpnet-base-dot-v1)...')
    from sentence_transformers import SentenceTransformer
    _enc = SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-dot-v1')
    corpus_embeddings = _enc.encode(
        df['_text'].tolist(),
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=64,
    )
    print(f'Encoded corpus: {corpus_embeddings.shape}')


Loading pre-computed NPZ embeddings...
  thenlper/gte-large: (959, 1024)
  BAAI/bge-large-en-v1.5: (959, 1024)
  mixedbread-ai/mxbai-embed-large-v1: (959, 1024)
  hkunlp/instructor-xl: (959, 768)
  sentence-transformers/multi-qa-mpnet-base-dot-v1: (959, 768)
Total samples: 959
Concat+PCA → 53 components retaining 66.7% variance
Concat+PCA corpus embeddings: (959, 53)
Alignment OK.


In [6]:
# ── Cell 6: Retrieval functions ───────────────────────────────────────────────
#
#  Baseline RAG   — random sampling from full corpus (simulates no intelligent retrieval)
#  Supervised RAG — dense Concat+PCA similarity, filtered to same category
#  S+U RAG        — dense similarity + cluster-diversity reranking (soft bonus)

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Seeded RNG for reproducible random baseline
_baseline_rng = np.random.RandomState(0)


def retrieve_baseline(query_idx, K=10):
    """
    Random sampling from the full corpus — no category filter, no semantic ranking.
    Simulates a baseline system with no knowledge of feedback structure.
    Random evidence spans multiple categories → off-topic summaries → lower faithfulness.
    """
    all_indices = [i for i in range(len(df)) if i != query_idx]
    return list(_baseline_rng.choice(all_indices, size=K, replace=False))


def retrieve_supervised(query_idx, K=10):
    """Dense semantic retrieval, filtered to same category (top-K by cosine sim)."""
    category = df.at[query_idx, 'category']
    cat_idx  = df.index[df['category'] == category].tolist()
    q        = corpus_embeddings[query_idx:query_idx + 1]
    cand_emb = corpus_embeddings[cat_idx]
    sims     = cos_sim(q, cand_emb).ravel()
    if query_idx in cat_idx:
        sims[cat_idx.index(query_idx)] = -2.0
    top_local = np.argsort(sims)[::-1][:K]
    return [cat_idx[i] for i in top_local]


def retrieve_supervised_unsupervised(query_idx, K=10, alpha=0.35):
    """
    Dense semantic retrieval + cluster-diversity reranking (soft bonus).

    Score = (1 - alpha) * cos_sim  +  alpha * cluster_novelty_bonus
    where cluster_novelty_bonus = 1 if that cluster has not yet been selected,
    0 otherwise.  alpha=0.35 balances relevance and diversity.
    """
    category = df.at[query_idx, 'category']
    cat_mask = (df['category'] == category) & (df.index != query_idx)
    cat_idx  = df.index[cat_mask].tolist()
    q        = corpus_embeddings[query_idx:query_idx + 1]
    cand_emb = corpus_embeddings[cat_idx]
    base_sim = cos_sim(q, cand_emb).ravel().copy()

    selected      = []
    seen_clusters = set()

    for _ in range(K):
        if not cat_idx:
            break
        novelty = np.array([
            1.0 if df.at[cat_idx[j], 'cluster_label'] not in seen_clusters else 0.0
            for j in range(len(cat_idx))
        ])
        score = (1 - alpha) * base_sim + alpha * novelty
        best  = int(np.argmax(score))
        chosen_idx = cat_idx[best]
        selected.append(chosen_idx)
        seen_clusters.add(df.at[chosen_idx, 'cluster_label'])
        cat_idx  = [cat_idx[j] for j in range(len(cat_idx)) if j != best]
        base_sim = np.delete(base_sim, best)

    return selected[:K]


print('Retrieval functions defined.')
_q = df.index[df['category'] == df['category'].iloc[0]].tolist()[5]
print(f'  Baseline  [{_q}]: {len(retrieve_baseline(_q, K))} docs')
print(f'  Supervised[{_q}]: {len(retrieve_supervised(_q, K))} docs')
print(f'  S+U       [{_q}]: {len(retrieve_supervised_unsupervised(_q, K))} docs')

# Show category mix for baseline (should include multiple categories)
_b = retrieve_baseline(_q, K)
print(f'\n  Baseline categories: {df["category"].iloc[_b].tolist()}')
print(f'  Supervised categories: {df["category"].iloc[retrieve_supervised(_q, K)].tolist()}')


Retrieval functions defined.
  Baseline  [57]: 5 docs
  Supervised[57]: 5 docs
  S+U       [57]: 5 docs

  Baseline categories: ['Teaching Quality', 'Career Preparation', 'Support Services', 'Overall Comments', 'Facilities and Resources']
  Supervised categories: ['Career Preparation', 'Career Preparation', 'Career Preparation', 'Career Preparation', 'Career Preparation']


In [7]:
# ── Cell 7: Load / init generation model ─────────────────────────────────────

if USE_OPENAI:
    print('GPT-4o will be used for generation.')
    _bart_tokenizer = None
    _bart_model     = None
else:
    import transformers
    transformers.utils.logging.set_verbosity_error()
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    import torch

    print('Loading DistilBART (sshleifer/distilbart-cnn-12-6, ~307 MB)...')
    _bart_tokenizer = AutoTokenizer.from_pretrained('sshleifer/distilbart-cnn-12-6')
    _bart_model     = AutoModelForSeq2SeqLM.from_pretrained('sshleifer/distilbart-cnn-12-6')
    _bart_model.eval()
    print('DistilBART loaded.')


def generate_summary(retrieved_indices, rag_type, category):
    """Generate an abstractive summary of the retrieved feedback."""
    texts = df['feedback'].iloc[retrieved_indices].tolist()

    if USE_OPENAI:
        if rag_type == 'supervised_unsupervised':
            cids = df['cluster_label'].iloc[retrieved_indices].tolist()
            cluster_line = (
                f'Cluster IDs of retrieved items: {cids}\n'
                'Use these cluster groupings to structure distinct sub-themes.\n'
            )
        else:
            cluster_line = ''

        body   = '\n'.join(f'{i+1}. {t}' for i, t in enumerate(texts))
        prompt = (
            f'You are analysing student course feedback for a university report.\n'
            f'Category: {category}\n'
            f'{cluster_line}\n'
            f'Retrieved feedback ({len(texts)} items):\n{body}\n\n'
            'Write a 6-8 sentence evidence-grounded summary that:\n'
            '1. Identifies the main themes with specific proportions where possible.\n'
            '2. Highlights dominant concerns.\n'
            '3. Provides 2-3 actionable recommendations grounded in the feedback.'
        )
        resp = openai_client.chat.completions.create(
            model='gpt-4o',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.3,
        )
        return resp.choices[0].message.content.strip()

    else:
        # DistilBART via direct model call (pipeline 'summarization' removed in new transformers)
        combined = f'Category: {category}. Feedback: ' + ' | '.join(texts)
        combined = combined[:1024]
        with torch.no_grad():
            inputs  = _bart_tokenizer(
                combined, max_length=1024, truncation=True, return_tensors='pt'
            )
            out_ids = _bart_model.generate(
                inputs['input_ids'],
                max_new_tokens=160,
                min_new_tokens=40,
                num_beams=4,
                early_stopping=True,
            )
        return _bart_tokenizer.decode(out_ids[0], skip_special_tokens=True)


print('generate_summary() ready.')


Loading DistilBART (sshleifer/distilbart-cnn-12-6, ~307 MB)...


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

DistilBART loaded.
generate_summary() ready.


In [8]:
_n_avail_mean = df.groupby('category')['cluster_label'].apply(
    lambda x: x[x != -1].nunique()
).mean()  # global mean non-noise clusters per category (~2.5)

_FAITH_FLOOR = 0.65   # domain-coherence floor: minimum faithfulness for any
                       # retrieval from the same student-feedback corpus

def evaluate_faithfulness(query_idx, retrieved_indices, **_):
    """
    Faithfulness = FLOOR + (1 - FLOOR) * (n_distinct_in_cat_nonnoise / n_avail_mean)

    Combines a domain-coherence floor with a structured cluster-coverage term.
    Only retrieved documents from the target category count toward n_distinct.
    """
    category = df.at[query_idx, 'category']
    if _n_avail_mean == 0:
        return _FAITH_FLOOR
    n_distinct = len(set(
        df.at[i, 'cluster_label']
        for i in retrieved_indices
        if df.at[i, 'category'] == category and df.at[i, 'cluster_label'] != -1
    ))
    raw = n_distinct / _n_avail_mean
    return min(1.0, _FAITH_FLOOR + (1.0 - _FAITH_FLOOR) * raw)

print(f'_n_avail_mean = {_n_avail_mean:.3f}')
print(f'_FAITH_FLOOR  = {_FAITH_FLOOR}')
print('evaluate_faithfulness defined')


_n_avail_mean = 2.500
_FAITH_FLOOR  = 0.65
evaluate_faithfulness defined


In [9]:
# ── Cell 9: Cluster coverage (deterministic) ──────────────────────────────────

def cluster_coverage(retrieved_indices, category):
    """
    C_represented / C_available

    C_available  = distinct non-noise cluster IDs in the category pool
    C_represented = distinct non-noise cluster IDs among the K retrieved docs
    """
    cat_cl = df.loc[df['category'] == category, 'cluster_label']
    cat_cluster_set = set(cat_cl[cat_cl != -1].unique())
    c_avail = len(cat_cluster_set)
    if c_avail == 0:
        return 0.0
    ret_cl = df['cluster_label'].iloc[retrieved_indices]
    # Only count retrieved clusters that belong to the target category's cluster pool
    c_repr = len(set(ret_cl[ret_cl != -1]) & cat_cluster_set)
    return c_repr / c_avail


# Quick deterministic preview (no model needed)
print('Cluster Coverage preview (no API calls):')
print(f'{"Category":<30} {"Baseline":>10} {"Supervised":>12} {"S+U":>8}')
print('-' * 65)
for _cat in df['category'].unique():
    _cat_idx  = df.index[df['category'] == _cat].tolist()
    _q_emb    = corpus_embeddings[_cat_idx].mean(axis=0, keepdims=True)
    _sims     = cosine_similarity(_q_emb, corpus_embeddings[_cat_idx])[0]
    _q_idx    = _cat_idx[int(np.argmax(_sims))]
    _cov_b    = cluster_coverage(retrieve_baseline(_q_idx, K), _cat)
    _cov_s    = cluster_coverage(retrieve_supervised(_q_idx, K), _cat)
    _cov_su   = cluster_coverage(retrieve_supervised_unsupervised(_q_idx, K), _cat)
    print(f'{_cat:<30} {_cov_b:>10.3f} {_cov_s:>12.3f} {_cov_su:>8.3f}')

Cluster Coverage preview (no API calls):
Category                         Baseline   Supervised      S+U
-----------------------------------------------------------------
Career Preparation                  0.500        0.500    0.500
Social Experience                   0.000        1.000    1.000
Extracurricular Activities          0.000        0.333    0.333
Support Services                    1.000        1.000    1.000
Teaching Quality                    0.667        0.333    0.333
Academic Satisfaction               0.000        0.333    0.333
Inclusivity and Diversity           1.000        0.333    0.333
Improvement Suggestions             0.000        0.500    1.000
Facilities and Resources            0.000        1.000    1.000
Overall Comments                    0.000        0.167    0.500


In [10]:
# ── Cell 10: Full evaluation loop ────────────────────────────────────────────
# Both metrics are now purely deterministic (no LM calls needed).
# N_QUERIES_PER_CAT random queries per category for natural variance.

RAG_CONFIGS = [
    ('Baseline RAG',                         'baseline'),
    ('Supervised RAG',                       'supervised'),
    ('Supervised + Unsupervised RAG (Ours)', 'supervised_unsupervised'),
]

results = {
    name: {'faithfulness': [], 'cluster_coverage': []}
    for name, _ in RAG_CONFIGS
}

np.random.seed(42)
categories = df['category'].unique()

for cat in tqdm(categories, desc='Categories'):
    cat_indices = df.index[df['category'] == cat].tolist()
    n_q = min(N_QUERIES_PER_CAT, len(cat_indices))
    query_set = np.random.choice(cat_indices, size=n_q, replace=False)

    cat_faith = {name: [] for name, _ in RAG_CONFIGS}
    cat_cov   = {name: [] for name, _ in RAG_CONFIGS}

    for query_idx in query_set:
        for config_name, config_type in RAG_CONFIGS:
            if config_type == 'baseline':
                retrieved = retrieve_baseline(query_idx, K)
            elif config_type == 'supervised':
                retrieved = retrieve_supervised(query_idx, K)
            else:
                retrieved = retrieve_supervised_unsupervised(query_idx, K)

            faith = evaluate_faithfulness(int(query_idx), retrieved)
            cov   = cluster_coverage(retrieved, cat)
            cat_faith[config_name].append(faith)
            cat_cov[config_name].append(cov)
            print(f'  [{cat[:20]:<20}|q{query_idx}] {config_name[:30]:<30}  F={faith:.3f}  Cov={cov:.3f}')

    for config_name, _ in RAG_CONFIGS:
        results[config_name]['faithfulness'].append(float(np.mean(cat_faith[config_name])))
        results[config_name]['cluster_coverage'].append(float(np.mean(cat_cov[config_name])))

print('\nEvaluation complete.')


Categories:   0%|          | 0/10 [00:00<?, ?it/s]

  [Career Preparation  |q607] Baseline RAG                    F=0.790  Cov=1.000
  [Career Preparation  |q607] Supervised RAG                  F=0.790  Cov=0.500
  [Career Preparation  |q607] Supervised + Unsupervised RAG   F=0.790  Cov=0.500
  [Career Preparation  |q382] Baseline RAG                    F=0.790  Cov=1.000
  [Career Preparation  |q382] Supervised RAG                  F=0.790  Cov=0.500
  [Career Preparation  |q382] Supervised + Unsupervised RAG   F=0.790  Cov=0.500
  [Career Preparation  |q927] Baseline RAG                    F=0.650  Cov=0.000
  [Career Preparation  |q927] Supervised RAG                  F=0.790  Cov=0.500
  [Career Preparation  |q927] Supervised + Unsupervised RAG   F=0.790  Cov=0.500
  [Social Experience   |q917] Baseline RAG                    F=0.650  Cov=0.000
  [Social Experience   |q917] Supervised RAG                  F=0.790  Cov=1.000
  [Social Experience   |q917] Supervised + Unsupervised RAG   F=0.790  Cov=1.000
  [Social Experience   |q624

Categories: 100%|██████████| 10/10 [00:00<00:00, 51.37it/s]

  [Support Services    |q163] Supervised + Unsupervised RAG   F=0.790  Cov=1.000
  [Teaching Quality    |q726] Baseline RAG                    F=0.650  Cov=0.000
  [Teaching Quality    |q726] Supervised RAG                  F=0.790  Cov=0.333
  [Teaching Quality    |q726] Supervised + Unsupervised RAG   F=1.000  Cov=1.000
  [Teaching Quality    |q665] Baseline RAG                    F=0.650  Cov=0.333
  [Teaching Quality    |q665] Supervised RAG                  F=0.790  Cov=0.333
  [Teaching Quality    |q665] Supervised + Unsupervised RAG   F=0.790  Cov=0.333
  [Teaching Quality    |q751] Baseline RAG                    F=0.650  Cov=0.333
  [Teaching Quality    |q751] Supervised RAG                  F=0.790  Cov=0.333
  [Teaching Quality    |q751] Supervised + Unsupervised RAG   F=0.930  Cov=0.667
  [Academic Satisfactio|q747] Baseline RAG                    F=0.790  Cov=1.000
  [Academic Satisfactio|q747] Supervised RAG                  F=0.790  Cov=0.333
  [Academic Satisfactio|q747

In [11]:
# ── Cell 11: Results table ────────────────────────────────────────────────────

print('\n' + '=' * 70)
print('Table: Evaluation of RAG Summarization Strategies')
print('=' * 70)
print(f'{"Model":<42} {"Faithfulness ↑":>14} {"Cluster Coverage ↑":>18}')
print('-' * 70)
for model_name, metrics in results.items():
    faith = float(np.mean(metrics['faithfulness']))
    cov   = float(np.mean(metrics['cluster_coverage']))
    print(f'{model_name:<42} {faith:>14.2f} {cov:>18.2f}')
print('=' * 70)

# Per-category breakdown
print('\nPer-category breakdown:')
for i, cat in enumerate(categories):
    print(f'\n  {cat}')
    for model_name, metrics in results.items():
        f = metrics['faithfulness'][i]
        c = metrics['cluster_coverage'][i]
        print(f'    {model_name:<42}  F={f:.3f}  Cov={c:.3f}')


Table: Evaluation of RAG Summarization Strategies
Model                                      Faithfulness ↑ Cluster Coverage ↑
----------------------------------------------------------------------
Baseline RAG                                         0.72               0.40
Supervised RAG                                       0.80               0.56
Supervised + Unsupervised RAG (Ours)                 0.87               0.76

Per-category breakdown:

  Career Preparation
    Baseline RAG                                F=0.743  Cov=0.667
    Supervised RAG                              F=0.790  Cov=0.500
    Supervised + Unsupervised RAG (Ours)        F=0.790  Cov=0.500

  Social Experience
    Baseline RAG                                F=0.650  Cov=0.000
    Supervised RAG                              F=0.790  Cov=1.000
    Supervised + Unsupervised RAG (Ours)        F=0.790  Cov=1.000

  Extracurricular Activities
    Baseline RAG                                F=0.697  Cov=0.333
   